# Collapsing Calo Energy Deposits from Duplicated Particles

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm
import h5py

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch

## Roadmap

1. Load a full-truth calo event
2. Load a reduced-truth calo event
3. Implement a particle aggregation function
4. Re-write the calo hit and calo contribution collections after aggregation
5. Prototype a re-writing of the edm4hep file, with the collapsed calo deposits

## Minimal Loading

In [26]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root"
event = EDM4hepEvent(edm_input_file, event_index=0)

Augmenting particles
Ready with tracker hits Index(['cellID', 'time', 'x', 'y', 'z', 'particle_id', 'detector', 'r', 'R',
       'phi', 'theta', 'eta'],
      dtype='object')
Augmenting particle hit counts with tracker hits


In [ ]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v7/runs/0/edm4hep.root"
event2 = EDM4hepEvent(edm_input_file, event_index=0)

In [4]:
print(f"""
Number of calo hits: {len(event.get_calo_hits_df())}
Number of particles: {len(event.get_particles_df())}
Number of tracker hits: {len(event.get_tracker_hits_df())}
""")


Number of calo hits: 1243953
Number of particles: 880327
Number of tracker hits: 240622



In [27]:
calo_hits_df = event.get_calo_hits_df()

In [28]:
calo_contributions_df = event.get_calo_contributions_df()

In [29]:
calo_hits_df

,cellID,energy,x,y,z,contribution_begin,contribution_end,detector,r,R,phi,theta,eta
0,104145539527933968,5.172027e-04,719.657898,-1058.644897,1881.900024,0,14,ECalBarrelCollection,1280.092407,2276.001709,-0.973762,0.597322,1.178078
1,50102494323496976,1.691220e-06,545.476929,1156.974487,902.700012,14,15,ECalBarrelCollection,1279.114990,1565.567749,1.130233,0.956240,0.657348
2,18278703426723737616,1.027635e-07,1459.449951,-102.000000,-3049.800049,15,16,ECalBarrelCollection,1463.010010,3382.554932,-0.069776,2.694311,-1.480845
3,18577387126204432,5.202330e-05,-1162.829468,-531.341553,336.600006,16,17,ECalBarrelCollection,1278.474243,1322.042236,-2.712984,1.313356,0.260332
4,18577387126237200,5.268137e-05,-1167.495117,-533.274109,336.600006,17,18,ECalBarrelCollection,1283.520996,1326.923462,-2.713125,1.314324,0.259331
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243948,12947956309262612,2.090424e-04,466.364319,1499.801392,3749.500000,6038419,6038421,HCalEndcapCollection,1570.636841,4065.175293,1.269324,0.396686,1.604522
1243949,12103535674065172,3.217657e-04,513.346008,1417.383423,3698.500000,6038421,6038422,HCalEndcapCollection,1507.481323,3993.920654,1.223313,0.387035,1.629795
1243950,9570265178603796,2.114068e-05,595.443970,1158.424194,3647.500000,6038422,6038423,HCalEndcapCollection,1302.497681,3873.081055,1.096002,0.342980,1.753360
1243951,10696160790479124,1.477808e-08,542.609558,1270.265625,3647.500000,6038423,6038424,HCalEndcapCollection,1381.303711,3900.289307,1.167096,0.362009,1.698227


In [30]:
calo_contributions_df

,energy,time,particle_id,cellID,x,y,z,detector
0,6.181428e-05,367.544739,84496,0,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
1,9.740763e-05,367.545288,84496,0,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
2,7.868026e-06,367.551208,84496,0,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
3,7.590641e-06,367.551300,84496,0,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
4,5.001049e-06,367.551422,84496,0,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
...,...,...,...,...,...,...,...,...
6038421,3.217657e-04,52.863934,879566,211086,513.346008,1417.383423,3698.500000,HCalEndcapCollection
6038422,2.114068e-05,229.844788,879422,211087,595.443970,1158.424194,3647.500000,HCalEndcapCollection
6038423,1.477808e-08,5697.791504,879422,211088,542.609558,1270.265625,3647.500000,HCalEndcapCollection
6038424,2.523652e-09,5751.806152,879422,211089,548.462280,1240.842163,3647.500000,HCalEndcapCollection


In [31]:
calo_contributions_df['cellID'].isin(calo_hits_df['cellID']).sum()

np.int64(0)

In [15]:
# We are going to aggregate duplicated contributions from each particle to each cell

aggregated_calo_contributions_df = calo_contributions_df.groupby(["cellID", "particle_id"]).agg({"energy": "sum", "time": "mean"}).reset_index()

In [23]:
calo_hits_df

,cellID,energy,x,y,z,contribution_begin,contribution_end,detector,r,R,phi,theta,eta
0,104145539527933968,5.172027e-04,719.657898,-1058.644897,1881.900024,0,14,ECalBarrelCollection,1280.092407,2276.001709,-0.973762,0.597322,1.178078
1,50102494323496976,1.691220e-06,545.476929,1156.974487,902.700012,14,15,ECalBarrelCollection,1279.114990,1565.567749,1.130233,0.956240,0.657348
2,18278703426723737616,1.027635e-07,1459.449951,-102.000000,-3049.800049,15,16,ECalBarrelCollection,1463.010010,3382.554932,-0.069776,2.694311,-1.480845
3,18577387126204432,5.202330e-05,-1162.829468,-531.341553,336.600006,16,17,ECalBarrelCollection,1278.474243,1322.042236,-2.712984,1.313356,0.260332
4,18577387126237200,5.268137e-05,-1167.495117,-533.274109,336.600006,17,18,ECalBarrelCollection,1283.520996,1326.923462,-2.713125,1.314324,0.259331
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243948,12947956309262612,2.090424e-04,466.364319,1499.801392,3749.500000,6038419,6038421,HCalEndcapCollection,1570.636841,4065.175293,1.269324,0.396686,1.604522
1243949,12103535674065172,3.217657e-04,513.346008,1417.383423,3698.500000,6038421,6038422,HCalEndcapCollection,1507.481323,3993.920654,1.223313,0.387035,1.629795
1243950,9570265178603796,2.114068e-05,595.443970,1158.424194,3647.500000,6038422,6038423,HCalEndcapCollection,1302.497681,3873.081055,1.096002,0.342980,1.753360
1243951,10696160790479124,1.477808e-08,542.609558,1270.265625,3647.500000,6038423,6038424,HCalEndcapCollection,1381.303711,3900.289307,1.167096,0.362009,1.698227


In [22]:
calo_contributions_df[["cellID", "x", "y", "z", "detector"]].drop_duplicates(subset=["cellID"])

,cellID,x,y,z,detector
0,0,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
14,1,545.476929,1156.974487,902.700012,ECalBarrelCollection
15,2,1459.449951,-102.000000,-3049.800049,ECalBarrelCollection
16,3,-1162.829468,-531.341553,336.600006,ECalBarrelCollection
17,4,-1167.495117,-533.274109,336.600006,ECalBarrelCollection
...,...,...,...,...,...
4713459,696376,82.810852,760.059937,3207.449951,ECalEndcapCollection
4713485,696377,661.298340,1119.524292,3394.300049,ECalEndcapCollection
4713493,696378,680.179565,-909.886780,3212.500000,ECalEndcapCollection
4713494,696379,199.069534,985.981934,3217.550049,ECalEndcapCollection


In [16]:
aggregated_calo_contributions_df

,cellID,particle_id,energy,time
0,0,53981,9.529331e-08,1600.755859
1,0,69135,2.574026e-05,94.866234
2,0,77728,1.878036e-06,444.325958
3,0,84490,1.456107e-03,14.045131
4,0,84493,1.570688e-05,1518.448730
...,...,...,...,...
2048408,696376,350,3.151920e-04,14.486650
2048409,696377,880287,1.880405e-04,8192.545898
2048410,696378,350,4.472489e-08,773.360901
2048411,696379,880276,2.476250e-04,25.576183


In [17]:
# reset the index, and merge back to get x, y, z, and detector
aggregated_calo_contributions_df = aggregated_calo_contributions_df.merge(calo_contributions_df[["cellID", "x", "y", "z", "detector"]], on="cellID", how="left")
aggregated_calo_contributions_df

,cellID,particle_id,energy,time,x,y,z,detector
0,0,53981,9.529331e-08,1600.755859,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
1,0,53981,9.529331e-08,1600.755859,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
2,0,53981,9.529331e-08,1600.755859,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
3,0,53981,9.529331e-08,1600.755859,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
4,0,53981,9.529331e-08,1600.755859,719.657898,-1058.644897,1881.900024,ECalBarrelCollection
...,...,...,...,...,...,...,...,...
38566134,696379,880276,2.476250e-04,25.576183,199.069534,985.981934,3217.550049,ECalEndcapCollection
38566135,696379,880276,2.476250e-04,25.576183,199.069534,985.981934,3217.550049,ECalEndcapCollection
38566136,696379,880276,2.476250e-04,25.576183,199.069534,985.981934,3217.550049,ECalEndcapCollection
38566137,696379,880276,2.476250e-04,25.576183,199.069534,985.981934,3217.550049,ECalEndcapCollection


In [ ]:
# reset the index, and merge back to get 
aggregated_calo_contributions_df

energy
cellID particle_id              
0      53981        9.529331e-08
       69135        2.574026e-05
       77728        1.878036e-06
       84490        1.456107e-03
       84493        1.570688e-05
...                          ...
696376 350          3.151920e-04
696377 880287       1.880405e-04
696378 350          4.472489e-08
696379 880276       2.476250e-04
696380 880276       5.677792e-08

[2048413 rows x 1 columns]